![Cabecera](../../assets/cabecera_sql.png)

# SQL en Python: Gestion BD (II)

En esta sesión completamos nuestra pequeña inmersión en el mundo del SQL y las bases de datos relacionales. Lo primero es volver a crearnos nuestra base de datos de la sesión anterior. Aquí te dejo las celdas preparadas: 

In [1]:
import pandas as pd
import sqlite3 

In [2]:
connection = sqlite3.connect("gestion_sesion.db")

In [3]:
cursor_gestion = connection.cursor()

In [4]:
query_create = '''
CREATE TABLE Master_Class (
ID       INT PRIMARY KEY, -- nombre tipo_de_dato restricción como hemos visto antes
NOMBRE   TEXT NOT NULL,
EDAD     INT NOT NULL,
CIUDAD   CHARS(50), -- Le decimos que este campo siempre tiene 50 caracteres, y los que no llene los rellenará con espacios
NOTAS    FLOAT
)
'''
cursor_gestion.execute(query_create)

In [5]:
# Usa este diccionario para no ir valor a valor:
datos = {
    "Luis": (24,"Madrid", 8.5),
    "Ana": (32,"Lugo", 6.25),
    "Juan": (35, "Bilbao", 5.55),
    "Nuria": (51, "Alicante", 9.75)
}
for indice,(nombre,valores) in enumerate(datos.items()):
    edad = valores[0]
    ciudad = valores[1]
    notas = valores[2]
    query = f"INSERT INTO Master_Class (ID,NOMBRE,EDAD,CIUDAD,NOTAS) VALUES ({indice},'{nombre}',{edad},'{ciudad}',{notas})"
    cursor_gestion.execute(query)


In [6]:
connection.commit()

### Update

La sintaxis general de UPDATE es:

```sql
UPDATE nombre_tabla
SET columna1 = valor1, columna2 = valor2, ...
WHERE condicion;
```

- **nombre_tabla:** Es el nombre de la tabla donde se realizarán las modificaciones.
- **SET columna1 = valor1, columna2 = valor2, ... :** Aquí se listan las columnas que se van a actualizar y los nuevos valores que se les asignarán. Puedes actualizar una o varias columnas a la vez.
- **WHERE condicion:** Especifica qué registros deben ser actualizados. La condición puede ser cualquier expresión lógica válida. Si omites la cláusula WHERE, todos los registros en la tabla serán actualizados, lo cual puede tener consecuencias no deseadas.

Modifiquemos en nuestra tabla la edad de Nuria por 48 y comprobemos el resultado utlizando `read_sql`

In [7]:
query = '''
UPDATE Master_Class
SET edad = 48
WHERE nombre = "Nuria"
'''
cursor_gestion.execute(query)
pd.read_sql("SELECT * FROM Master_Class", connection)

,ID,NOMBRE,EDAD,CIUDAD,NOTAS
0,0,Luis,24,Madrid,8.50
1,1,Ana,32,Lugo,6.25
2,2,Juan,35,Bilbao,5.55
3,3,Nuria,48,Alicante,9.75


Pero ojo no nos dejemos engañar, necesitamos hacer el commit igualmente para que el cambio se grabe en la base de datos:

In [8]:
connection.commit()

Si hubieramos intentado actualizar alguna columna incumpliendo las restricciones con las que fueron creadas hubiera dado error.

### Delete

DELETE nos permite borrar filas de las tablas, y sigue esta sintaxis:

```sql
DELETE FROM nombre_tabla WHERE condicion;
```

- **nombre_tabla:** Es el nombre de la tabla de la cual se eliminarán los registros.
- **condicion:** Especifica qué registros deben ser eliminados. Puede ser cualquier expresión lógica válida.

Si omites la cláusula WHERE en una instrucción DELETE, todos los registros de la tabla especificada serán eliminados, lo cual debe hacerse con precaución.

Borremos las filas de los alumnos con menos de un 7:

In [9]:
query = '''
DELETE FROM Master_Class'''

cursor_gestion.execute(query)

In [10]:
pd.read_sql("SELECT* FROM Master_Class", connection)

,ID,NOMBRE,EDAD,CIUDAD,NOTAS


Eh, que susto, tranquilidad, como con `INSERT` y `UPDATE` es necesario hacer el commit para que los cambios se hagan efectivos. En este caso cierra la conexión a la base de datos y abre una nueva.

In [11]:
connection.close()
connection = sqlite3.connect("gestion_sesion.db")
cursor_gestion = connection.cursor()

In [12]:
pd.read_sql("SELECT * FROM Master_Class", connection)

,ID,NOMBRE,EDAD,CIUDAD,NOTAS
0,0,Luis,24,Madrid,8.50
1,1,Ana,32,Lugo,6.25
2,2,Juan,35,Bilbao,5.55
3,3,Nuria,48,Alicante,9.75


 Y ahora sí borremos esas filas:

In [13]:
query = '''
DELETE FROM Master_Class
WHERE notas < 7'''

cursor_gestion.execute(query)
pd.read_sql("SELECT* FROM Master_Class", connection)

,ID,NOMBRE,EDAD,CIUDAD,NOTAS
0,0,Luis,24,Madrid,8.50
1,3,Nuria,48,Alicante,9.75


De nuevo, hay que hacer el commit, hagámoslo:


In [14]:
connection.commit()

### Borrado de columnas

Igual que en pandas tenemos el método `drop` con su argumento columns para borrar columnas en SQL tenemos:


```sql
ALTER TABLE nombre_tabla DROP COLUMN nombre_columna;
```

- **nombre_tabla:** Es el nombre de la tabla de la que deseas eliminar una columna.
- **nombre_columna:** Es el nombre de la columna que deseas eliminar.



**Nota Importante:**
- Algunos sistemas de bases de datos pueden requerir modificadores adicionales o tener restricciones específicas para eliminar columnas. Por ejemplo, si otras tablas dependen de la columna que estás intentando eliminar (a través de claves foráneas, por ejemplo), es posible que necesites modificar o eliminar esas dependencias antes de poder eliminar la columna.
- Siempre es una buena práctica hacer una copia de seguridad de tu base de datos antes de realizar cambios estructurales como eliminar columnas.

Probemos a borrar la columna "edad":

In [15]:
query = '''
ALTER TABLE Master_Class
DROP COLUMN edad '''

cursor_gestion.execute(query)

Pero recuerdad que si no haces el commit tampoco se grabará el cambio, ejecuta lo siguiente:

In [16]:
connection.commit()

Tienes que hacer el commit después del ALTER/DROP.

### Borrado de tablas

Para terminar la sesión y el grupo dedicado a SQL y bases de datos, veamos como borrar una tabla:


```sql
DROP TABLE nombre_tabla;
```

**Algunas consideraciones importantes:**

1. **Cuidado con los Datos:** Al usar `DROP TABLE`, la tabla y todos sus datos se eliminan permanentemente. No hay forma de deshacer esta acción en la mayoría de los sistemas de gestión de bases de datos. Asegúrate de que realmente deseas eliminar la tabla y de que has realizado una copia de seguridad de los datos si es necesario.

2. **Dependencias:** Si la tabla está referenciada por otras tablas a través de claves foráneas u otros mecanismos de integridad referencial, es posible que primero debas eliminar o modificar estas dependencias.

3. **Sintaxis Específica del SGBD:** La sintaxis básica de `DROP TABLE` es bastante uniforme en los diferentes sistemas de gestión de bases de datos (SGBD), pero algunos pueden ofrecer opciones adicionales. Por ejemplo, en algunos SGBD puedes usar `DROP TABLE IF EXISTS nombre_tabla;` para evitar errores si la tabla no existe.

4. **Permisos:** Necesitas tener los permisos adecuados en la base de datos para eliminar tablas. Si no tienes los permisos necesarios, la operación fallará.

In [17]:
query = '''
DROP TABLE Master_Class
'''
cursor_gestion.execute(query)

Veamos que pasa con la master table de Sqlite donde debería haber desaparecido:

In [18]:
pd.read_sql("SELECT Name FROM sqlite_master WHERE type == 'table'", connection)

,name


De nuevo tendrás que hacer commit para asegurarte de que se borra de verdad. Ojo, en algunos gestores al cerrar el gestor o una base de datos se hace un commit automático así que no te fíes después de haber borrado accidentalmente cierra la base de datos pero evitando que se hagan commits.